<a href="https://colab.research.google.com/github/kast20001/Map_of_skills/blob/main/lab_01_analysis_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Лабораторная работа №1

Вы - аналитик небольшого интернет-магазина. Из CRM выгрузили все заказы за январь-апрель 2024 года в файл `lab_01_dataset.csv`. Руководитель хочет знать, сколько магазин заработал, какие категории товаров приносят больше денег и довольны ли покупатели.

Проблема в том, что заказы попадают в CRM из разных мест: с сайта, из мобильного приложения и вручную от менеджеров (часть - через Excel). Поэтому одни и те же по смыслу значения записаны по-разному. Прежде чем считать хоть одну цифру, нужно понять, **можно ли этим данным доверять**, и привести их в порядок.

## Правила

- Файл `lab_01_dataset.csv` должен лежать в той же папке, что и ноутбук. **Сам CSV не редактируйте** - исходные данные никогда не меняем, все исправления делаются только кодом.
- Код пишите в пустых ячейках под заданиями. Ответы на вопросы - в ячейках **«Ваш ответ»** (1-4 предложения, своими словами).
- Используйте инструменты, изученные на лекциях 1-2. Задача работы - закрепить изученное, а не найти самый короткий способ.
- Там, где задание просит *сначала предсказать* результат, сначала запишите предсказание, потом запускайте код.
- Перед сдачей: **Kernel -> Restart Kernel and Run All** /  **Среда выполнения -> Перезапустить сеанс и выполнить весь код**. Если что-то сломалось - значит, работа была сломана и раньше (скрытое состояние ноутбука).
- Вы должны уметь пояснить **любую** строку своего решения.
- Функция `parse_price`, словари `city_map` и `status_map` и функция `load_orders` из этой работы понадобятся на практическом занятии, которое проходит после лабораторной. Сохраните ноутбук.

> **Про версии pandas.** В pandas 3.x текстовые столбцы по умолчанию получают тип `str`, а не `object`. Для этой работы разница непринципиальна: и `object`, и `str` в числовом по смыслу столбце - признак проблемы. Везде, где в тексте написано `object`, в pandas 3 вы можете увидеть `str`.

## Описание данных (из документации CRM)

| Столбец | Что означает |
|---|---|
| `order_id` | номер заказа, уникальный |
| `order_date` | дата оформления заказа |
| `customer_id` | идентификатор покупателя; у гостевых заказов (без регистрации) отсутствует |
| `city` | город доставки |
| `product_name` | название товара |
| `category` | категория товара |
| `price` | цена за единицу товара, руб. |
| `quantity` | количество единиц товара в заказе |
| `discount` | скидка в процентах; **пустое значение означает, что скидка не применялась** |
| `is_paid` | оплачен ли заказ (да / нет) |
| `status` | статус заказа: доставлен, в пути, отменен, возврат |
| `delivery_date` | дата доставки; у недоставленных заказов отсутствует |
| `weight_kg` | вес одной единицы товара, кг; у электронных товаров отсутствует |
| `rating` | оценка покупателя от 1 до 5; ставится не всегда |

## Содержание

0. Подготовка
1. Базовый Python: смотрим на сырые данные без pandas
2. numpy: тип элементов массива
3. Загрузка и первичный осмотр выгрузки
4. Диагностика: почему столбцы не числовые
5. Числовые столбцы
6. Даты
7. Строки, категории и логические значения
8. Новые признаки
9. Контроль результата
10. Итоговое самостоятельное задание

---
## Часть 0. Подготовка

Выполните ячейку ниже. Если возникла ошибка импорта - установите недостающую библиотеку в свое окружение.

In [ ]:
import copy

import numpy as np
import pandas as pd

DATA_PATH = "lab_01_dataset.csv"

print("numpy", np.__version__)
print("pandas", pd.__version__)

При желании вы можете обновить версию pandas, раскомментировав строку ниже. После запуска перезапустите сеанс.

In [ ]:
#!pip install -U pandas

---
## Часть 1. Базовый Python: смотрим на сырые данные без pandas

В этой части мы еще **не используем pandas** - только встроенные средства языка.

### 1.1. Файл как текст

**Задание.**

1. С помощью конструкции `with open(...)` прочитайте файл `DATA_PATH` целиком в строку `text` (кодировка `utf-8`).
2. Разбейте текст на список строк `lines`.
3. С помощью распаковки выделите заголовок и строки данных: `header, *data_lines = ...`.
4. Получите список названий столбцов `columns`.
5. Выведите количество строк данных, а затем для каждого столбца - его порядковый номер и название в формате `0: order_id` (используйте `enumerate` и f-строку).

> **Подсказка.** Пригодится метод строк `.split(...)`. Посмотрите, чем заканчивается файл: не окажется ли последним элементом списка пустая строка? Метод `.strip()` убирает пробельные символы по краям строки.

In [ ]:
# with open(...) as f:
#     text = ...

# lines = ...
# header, *data_lines = ...
# columns = ...

### 1.2. Сколько полей в строке?

Кажется, что CSV - это просто «значения через запятую». Проверим.

**Задание.**

1. Для каждой строки из `data_lines` посчитайте, на сколько частей она делится по запятой. Соберите все встретившиеся количества в **множество** с помощью set comprehension и выведите его.
2. С помощью list comprehension соберите строки, у которых число частей **не совпадает** с числом столбцов. Выведите, сколько таких строк, и напечатайте первые три из них.

In [ ]:
# Ваш код

**Вопрос 1.2.** Почему у большинства строк «полей» больше, чем столбцов в заголовке? Какие значения в этом виноваты? Можно ли надежно разбирать такой файл обычным `split(",")`?

**Ваш ответ:**

_..._

### 1.3. Множества и словари: сколько на самом деле городов и статусов?

Ниже - значения столбцов `city` и `status` первых 30 заказов. Выполните ячейку.

In [ ]:
raw_cities = ['Москва', 'Москва', 'Казань', 'Москва', 'МОСКВА', 'Москва', 'Екатеринбург', 'Москва', 'Санкт-Петербург', 'Москва', 'Санкт-Петербург', 'Москва', 'Москва', 'Казань', 'Казань', 'Москва', 'Москва', 'Москва', 'Москва', 'Новосибирск', 'Казань', ' Москва', 'Москва', 'Москва', 'Москва', 'Казань', 'Нижний Новгород', 'Москва', 'Санкт-Петербург', 'Москва']
raw_statuses = ['В пути', 'доставлен', 'доставлен', 'В пути', 'в пути', 'доставлен', 'доставлен', 'доставлен', 'доставлен', 'доставлен', 'доставлен', 'Доставлен', 'доставлен', 'в пути', 'доставлен', 'доставлен', 'доставлен', 'доставлен', 'доставлен', 'доставлен', 'доставлен', 'доставлен', 'доставлен', 'доставлен', 'доставлен', 'в пути', 'отменён', 'отменен', 'доставлен', 'доставлен']

**Задание 1.3.1.**

1. Сколько **различных написаний** городов в `raw_cities`? Выведите их.
2. С помощью set comprehension приведите написания к единому виду: убрать пробелы по краям и перевести в нижний регистр (`.lower()`). Сколько городов осталось на самом деле?
3. Создайте словарь `city_map`: *нормализованное написание -> каноническое название* (как принято в справочнике: `"Москва"`, `"Санкт-Петербург"`, `"Нижний Новгород"` и т.д.). Ключами должны быть все варианты из пункта 2. **Этот словарь понадобится в части 7.**
4. Циклом по нормализованным написаниям проверьте оператором `in`, что каждое из них есть среди ключей `city_map`. Выведите те, которых нет (должно получиться пусто).

In [ ]:
# Ваш код

**Задание 1.3.2.**

1. Посчитайте, сколько раз встречается каждый статус в `raw_statuses` **после нормализации** (strip + lower). Используйте словарь и цикл (без готовых функций подсчета). Выведите результат в виде `доставлен: 21`.
2. Создайте словарь `status_map`: *нормализованное написание -> каноническое значение* из документации (`"доставлен"`, `"в пути"`, `"отменен"`, `"возврат"`). Ключами должны быть все варианты, которые вы видите в выборке. **Словарь понадобится в части 7.**
3. Проверьте, что после перекодировки через `status_map` в выборке остались только значения из документации.

In [ ]:
# Ваш код

**Вопрос 1.3.** (а) Почему после нормализации все еще остались два разных «отмененных» статуса и что с этим делать? (б) Представьте, что в выгрузке 2 млн строк и для каждой нужно проверять, встречалось ли значение раньше. Где хранить уже встреченные значения - в списке или в множестве? Почему это важно?

**Ваш ответ:**

_..._

### 1.4. Функция разбора цены и обработка исключений

Ниже - значения столбца `price` первых 36 заказов, как они записаны в файле.

In [ ]:
raw_prices = ['590.00', '2499.00', '250,00', '790.00', '1190.00', '23\xa0990,00', '6790.00', '23990.00', '7 450 руб.', '590.00', '5990.00', '4 590,00', '3490.00', '8990.00', '3150.00', '23990.00', '1850.00', '790.00', '2\xa0190,00', '5990.00', '1 290,00', '3490.00', '6790.00', '7450.00', '7450.00', '250 руб.', '590.00', '2190.00', '590.00', '7 450,00', '6790.00', '590,00', '1290.00', '8990.00', 'н/д', '6790.00']

**Задание 1.4.1.** Выведите список `raw_prices` целиком (именно как список, через `print`) и внимательно рассмотрите значения.

**Вопрос 1.4.1.** Какие форматы записи цены вы видите? Что означает `\xa0` внутри некоторых значений и почему его не видно, если напечатать само значение?

In [ ]:
# Ваш код

**Ваш ответ:**

_..._

**Задание 1.4.2.** Напишите функцию `parse_price(raw)`, которая превращает строку с ценой в `float`, а если это невозможно - возвращает `None`. Функция должна:
- убирать пробелы по краям и пометку `руб.`;
- убирать разделители разрядов (обычный и неразрывный пробел);
- заменять десятичную запятую на точку;
- пытаться выполнить `float(...)` и перехватывать **только** ту ошибку, которая возникает при неудачном преобразовании строки в число.

> **Подсказка.** Методы строк `.strip()` и `.replace(старое, новое)`. Неразрывный пробел записывается как `"\xa0"`. Помните, что `except:` без указания типа - почти всегда ошибка проектирования.

In [ ]:
def parse_price(raw):
    ...

**Задание 1.4.3.**

1. Примените `parse_price` ко всем `raw_prices` (list comprehension) и получите список `parsed_prices`.
2. Посчитайте, сколько значений не удалось разобрать, и выведите долю таких значений в процентах с одним знаком после запятой с помощью f-строки (формат `:.1%`).
3. С помощью `zip` и `enumerate` выведите номер позиции и исходное значение для каждого неразобранного случая.
4. Соберите список успешно разобранных цен `valid_prices`, отсортируйте его (`sorted`) и распаковкой `cheapest, *rest = ...` получите самую низкую цену; самую высокую возьмите из `rest`.

In [ ]:
# Ваш код

---
## Часть 2. numpy: тип элементов массива

pandas хранит столбцы в массивах numpy. Если понимать, как устроен `ndarray`, многое в поведении pandas станет очевидным - в том числе появление `object`.

### 2.1. Тип массива определяется всеми элементами

**Задание 2.1.** **Сначала предскажите** (запишите в ячейку ответа ниже), какой тип элементов будет у каждого массива, затем создайте массивы и выведите для каждого `dtype`, `shape` и `ndim` (удобно - циклом по списку массивов с f-строкой):

```python
a1 = np.array([1, 2, 3, 4, 5])          # количества
a2 = np.array([590.0, 2499.0, 250.0])   # цены
a3 = np.array([1, 2, "3 шт"])           # количества «как в выгрузке»
a4 = np.array([1, 2, None])             # количество с пропуском
```

После этого попробуйте вычислить `a3 * 2` (перехватите ошибку через `try / except` с конкретным типом) и `a4 * 2`.

In [ ]:
# Ваш код

**Вопрос 2.1.** Что произошло с массивами `a3` и `a4`? Как это связано с тем, что столбец таблицы pandas может получить тип `object`, хотя «почти все» значения в нем - числа?

**Ваш ответ:**

_..._

---
## Часть 3. Загрузка и первичный осмотр выгрузки

### 3.1. Первое знакомство

**Задание 3.1.** Загрузите файл `DATA_PATH` в DataFrame `raw`. Выведите (каждое - отдельной командой или ячейкой):
- размер таблицы;
- первые 10 строк;
- типы столбцов;
- `info` с подсчетом памяти `memory_usage="deep"`;
- `describe()`.

In [ ]:
# Ваш код

In [ ]:
# Ваш код

In [ ]:
# Ваш код

In [ ]:
# Ваш код

**Вопрос 3.1.1.** Заполните таблицу для всех столбцов: какой тип определил pandas и каким тип должен быть **по смыслу** (используйте описание данных в начале ноутбука). Отметьте столбцы, где тип не соответствует смыслу.

| Столбец | Тип после загрузки | Тип по смыслу | Проблема? |
|---|---|---|---|
| `order_id` | | | |
| ... | | | |

**Вопрос 3.1.2.** Почему `describe()` показал статистику только по двум столбцам, хотя числовых по смыслу столбцов гораздо больше?

**Вопрос 3.1.3.** Какие столбцы содержат пропуски и сколько их? Для каких из них пропуски ожидаемы по описанию данных?

**Ваш ответ:**

_..._

### 3.2. Загадка `customer_id`

В столбце `customer_id` нет ни одного текстового значения, но его тип - `float64`, а не целочисленный.

**Задание 3.2.** Используя список `data_lines` из части 1, найдите (list comprehension) строки исходного файла, в которых встречается подстрока `"N/A"`, и выведите их. Затем выведите значения `customer_id` для этих заказов из `raw` (обратиться к ним можно, например, через булеву маску `raw["order_id"].isin([...])`). Номера заказов впишите вручную.

In [ ]:
# Ваш код

**Вопрос 3.2.** (а) Почему `customer_id` получил тип `float64`? (б) В файле для двух гостевых заказов написано `N/A`, а в столбце `price` встречается `н/д`. Почему `N/A` не помешал pandas распознать числа, а `н/д` - помешал? (в) Чем плох тип `float64` для идентификатора?

**Ваш ответ:**

_..._

### 3.3. Рабочая копия и индекс

**Задание 3.3.1.** Все преобразования будем делать в рабочей копии, а `raw` сохраним нетронутым для сравнения.

1. Проверьте, что номера заказов уникальны (сравните число различных значений с числом строк).
2. Создайте рабочую таблицу `df` - копию `raw`, в которой `order_id` сделан индексом.

In [ ]:
# Ваш код

**Вопрос 3.3.1.** Почему нельзя было просто написать `df = raw` и дальше менять `df`? Почему `order_id` - хороший кандидат в индекс?

**Ваш ответ:**

_..._

**Задание 3.3.2.**

1. Выведите заказ `10013` по метке и ту же строку по позиции.
2. Выведите цену заказа `10009` двумя способами: через `.loc` (метка + имя столбца) и через `.iloc` (позиции строки и столбца).
3. Выведите заказы с 10010 по 10015 через `.loc` и строки с позиции 9 по 15 через `.iloc`. Сравните количество строк.

In [ ]:
# Ваш код

In [ ]:
# Ваш код

In [ ]:
# Ваш код

**Вопрос 3.3.2.** Почему срезы в пункте 3 вернули разное количество строк? Какой из способов ведет себя как срезы обычных списков Python?

**Ваш ответ:**

_..._

---
## Часть 4. Диагностика: почему столбцы не числовые

### 4.1. Прямая попытка

**Задание 4.1.** Для столбцов `price`, `quantity`, `discount`, `weight_kg`, `rating` попробуйте (циклом по списку названий) привести столбец `df` к `float` методом `astype`. Ошибку перехватывайте с конкретным типом и выводите название столбца и текст ошибки. Результат преобразования в `df` пока **не записывайте**.

In [ ]:
# Ваш код

**Вопрос 4.1.** Какую информацию дала ошибка? Можно ли по ней понять **все** проблемы столбца?

**Ваш ответ:**

_..._

### 4.2. Функция поиска «плохих» значений

**Задание 4.2.** Напишите функцию `find_bad_values(s)`, которая принимает Series и возвращает **множество** значений, которые не удается интерпретировать как число. Пропуски (`NaN`) при этом плохими считать не нужно - они и так пропуски.

Примените функцию ко всем столбцам из 4.1 в цикле и для каждого выведите: название, количество различных плохих значений и само множество (или первые 10 значений из отсортированного списка, если их много).

> **Подсказка.** `pd.to_numeric(..., errors="coerce")` превращает все, что не удалось разобрать, в `NaN`. «Плохое» значение - то, что стало `NaN` после преобразования, **но не было** `NaN` до него.

In [ ]:
def find_bad_values(s):
    ...

**Вопрос 4.2.** Заполните таблицу: какие причины мешают каждому столбцу стать числовым и что с ними нужно сделать (удалить символы, заменить символ, считать пропуском и т.п.). Какие из «плохих» значений на самом деле содержат корректное число, а какие - действительно означают отсутствие данных?

| Столбец | Причины | Что делать |
|---|---|---|
| `price` | | |
| `quantity` | | |
| `discount` | | |
| `weight_kg` | | |
| `rating` | | |

**Ваш ответ:**

_..._

---
## Часть 5. Числовые столбцы

### 5.1. `raise` против `coerce`

**Задание 5.1.** Не очищая столбец `price`, преобразуйте его функцией `pd.to_numeric`:
1. с `errors="raise"` (перехватите ошибку и выведите ее текст);
2. с `errors="coerce"` - посчитайте, сколько пропусков получилось.

Сравните число пропусков с тем, сколько цен **на самом деле** неизвестно (см. 4.2). Результат в `df` не записывайте.

In [ ]:
# Ваш код

**Вопрос 5.1.** Чем отличаются режимы `errors="raise"` и `errors="coerce"`? Какой режим используется по умолчанию? Почему применять `coerce` к неочищенному столбцу опасно - сколько реальных цен мы бы потеряли и заметили бы мы это без специальной проверки?

**Ваш ответ:**

_..._

### 5.2. Очистка цены

**Задание 5.2.**

1. Создайте очищенную **текстовую** Series `price_text`: уберите пробелы по краям, пометку `руб.`, разделители разрядов (обычный и неразрывный пробел), замените десятичную запятую на точку.
2. Проверьте ее функцией `find_bad_values`: что осталось?
3. Выберите подходящий режим `errors`, преобразуйте `price_text` в число и запишите результат в `df["price"]`.
4. Проверьте результат: тип столбца, число пропусков (оно должно совпасть с количеством действительно неизвестных цен), `describe()`, отсутствие неположительных цен.

> **Подсказка.** У Series есть «строковые» методы, которые работают как методы обычных строк, но сразу для всего столбца: `s.str.strip()`, `s.str.replace(старое, новое)`, `s.str.lower()` и т.д. В лекции это первый пункт порядка выбора: «есть встроенный метод (`.str`, `.dt`, арифметика)? - используем его».

In [ ]:
# Ваш код

In [ ]:
# Ваш код

**Вопрос 5.2.** Какой режим `errors` вы выбрали на шаге 3 и почему теперь он безопасен? Что изменилось бы, если бы в следующей выгрузке появилось значение `"1.290,00"` (точка как разделитель разрядов)?

**Ваш ответ:**

_..._

### 5.3. Количество

**Задание 5.3.** Приведите `quantity` к целочисленному типу. Сначала посмотрите на множество всех значений столбца, затем очистите их так, чтобы **не потерять** ни одного реального количества. Перед финальным преобразованием убедитесь (`find_bad_values`), что неразбираемыми остались только действительно неизвестные значения.

Отдельно попробуйте привести числовой результат к обычному `int` (перехватите ошибку) и объясните, что произошло. Итоговый столбец должен быть целочисленным **с поддержкой пропусков**.

Проверьте: тип, число пропусков, отсутствие неположительных значений.

In [ ]:
# Ваш код

**Вопрос 5.3.** (а) Сколько реальных количеств было бы потеряно, если бы мы сразу применили `coerce` к неочищенному столбцу? (б) Почему `pd.to_numeric` вернул `float64`, хотя все количества целые? (в) Почему не подошел `int`, а `Int64` подошел?

**Ваш ответ:**

_..._

### 5.4. Скидка

**Задание 5.4.** Приведите `discount` к **доле** от 0 до 1 (например, `"15%"` -> `0.15`). Учтите описание данных: пустое значение означает, что скидка не применялась.

Подумайте, какой режим `errors` здесь уместнее. Замену пропусков выполните присваиванием по маске через `.loc`. Проверьте: тип, отсутствие пропусков, что все значения лежат в диапазоне [0, 1] (`between`), и выведите множество получившихся значений.

In [ ]:
# Ваш код

**Вопрос 5.4.** Почему здесь уместнее `errors="raise"`, а не `coerce`? Почему пропуски в `discount` можно заменить нулем, а пропуски в `price` - нельзя?

**Ваш ответ:**

_..._

### 5.5. Идентификатор покупателя

**Задание 5.5.** Приведите `customer_id` к целочисленному типу с поддержкой пропусков. Убедитесь, что число пропусков не изменилось и что значения не потеряли дробную часть (у исходных `float` значений она нулевая: проверьте через остаток от деления на 1).

In [ ]:
# Ваш код

---
## Часть 6. Даты

### 6.1. Дата заказа

В документации CRM сказано, что даты хранятся в формате `ДД.ММ.ГГГГ`.

**Задание 6.1.1.** Преобразуйте `order_date` функцией `pd.to_datetime`, указав формат из документации:
1. в режиме по умолчанию (перехватите ошибку и выведите текст);
2. с `errors="coerce"` - сохраните результат в переменную `dates`, посчитайте пропуски.

Выведите исходные значения `order_date` для **всех** строк, которые не удалось преобразовать (не считая исходных пропусков).

In [ ]:
# Ваш код

**Вопрос 6.1.1.** Разделите непреобразованные значения на группы: какие из них - корректные даты в другом формате, а какие - действительно ошибочные? Почему значение `31.04.2024` не преобразовалось, хотя его формат совпадает с документацией?

**Ваш ответ:**

_..._

**Задание 6.1.2.** Восстановите даты второго формата: преобразуйте **только** непреобразованные строки с форматом `"%Y-%m-%d"` и запишите результат в `dates` по маске через `.loc`. Затем запишите `dates` в `df["order_date"]`.

Проверьте: тип столбца; сколько осталось пропусков и каким исходным значениям они соответствуют (используйте `raw`, помня об индексе); что все даты лежат в периоде выгрузки (январь-апрель 2024) - например, с помощью `describe()` или `between`.

In [ ]:
# Ваш код

**Вопрос 6.1.2.** Почему pandas сам не распознал даты при загрузке файла? Какие пропуски в `order_date` остались, можно ли их восстановить и как вы поступили бы с такими заказами?

**Ваш ответ:**

_..._

---
## Часть 7. Строки, категории и логические значения

### 7.1. Города

**Задание 7.1.1.** Нормализуйте `city` (пробелы по краям, нижний регистр) и перекодируйте значения словарем `city_map` из части 1 методом `.map`. Посчитайте, сколько значений **не нашлось** в словаре, и выведите их множество.

In [ ]:
# Ваш код

**Вопрос 7.1.1.** Почему словарь, который полностью подходил к выборке в части 1, не подошел ко всей таблице? Что вернул `.map` для таких значений?

**Ваш ответ:**

_..._

**Задание 7.1.2.** Дополните `city_map` недостающими написаниями (добавьте ключи здесь, не редактируя ячейку части 1), повторите перекодировку и убедитесь, что ненайденных значений нет. Затем:

1. Запомните объем памяти столбца `city` до преобразования (`memory_usage(deep=True)`).
2. Запишите перекодированные значения в `df["city"]` с типом `category`.
3. Выведите объем памяти после преобразования, список категорий и первые 5 целочисленных кодов.
4. Выведите количество заказов в каждом городе (циклом по категориям и суммой булевой маски).

In [ ]:
# Ваш код

**Вопрос 7.1.2.** Как устроен тип `category` внутри и почему он подходит для столбца `city`? Во сколько раз уменьшилась память и почему на таблице в миллионы строк выигрыш будет еще заметнее?

**Ваш ответ:**

_..._

### 7.2. Статусы

**Задание 7.2.** Аналогично приведите `status` к типу `category`, используя `status_map` из части 1: нормализация -> `.map` -> проверка ненайденных значений -> при необходимости дополнение словаря -> запись в `df`. Выведите количество заказов в каждом статусе и убедитесь, что статусы совпадают с документацией.

In [ ]:
# Ваш код

### 7.3. Категория и название товара

**Задание 7.3.** Приведите `category` к типу `category`, а `product_name` - к строковому типу `string`. Выведите количество различных значений в каждом из столбцов и сравните с количеством строк.

In [ ]:
# Ваш код

**Вопрос 7.3.** Чем отличаются типы `object`, `string` и `category`? Почему для `category` выбран категориальный тип, а для `product_name` - строковый, хотя различных названий тоже немного?

**Ваш ответ:**

_..._

### 7.4. Оплата

**Задание 7.4.** Преобразуйте `is_paid` в логический тип:

1. Выведите множество значений столбца, нормализуйте их и перекодируйте словарем в `True` / `False`.
2. Приведите результат двумя способами - к `bool` и к `boolean` - и для каждого посчитайте количество оплаченных заказов (`True`) и пропусков.
3. Запишите в `df` правильный вариант.

In [ ]:
# Ваш код

**Вопрос 7.4.** Почему результаты различаются? Какой вариант верный и почему ошибка первого варианта особенно опасна?

**Ваш ответ:**

_..._

---
## Часть 8. Новые признаки

### 8.1. Выручка

**Задание 8.1.1.** Добавьте столбец `revenue` - выручку по заказу с учетом количества и скидки - **векторной операцией** над столбцами. Выведите тип получившегося столбца и число пропусков. Приведите `revenue` к обычному `float64` (он понадобится для функций numpy в 8.2) и проверьте тип еще раз.

In [ ]:
# Ваш код

**Вопрос 8.1.1.** Откуда в `revenue` взялись пропуски и почему их именно столько? Почему выручка получила тип `Float64` (с заглавной буквы)? Почему такой способ вычисления лучше, чем цикл `for i, row in df.iterrows(): ...`?

**Ваш ответ:**

_..._

**Задание 8.1.2.** По договоренности с бухгалтерией выручка по отмененным и возвращенным заказам должна быть равна 0. Коллега написал:

```python
df[df["status"] == "отменен"]["revenue"] = 0
```

**Сначала предскажите**, что произойдет. Затем посчитайте, у скольких отмененных заказов выручка не равна 0, выполните код коллеги и посчитайте еще раз. После этого напишите правильный вариант, который обнуляет выручку и у отмененных, и у возвращенных заказов, и проверьте результат.

In [ ]:
# Ваш код

In [ ]:
# Ваш код

**Вопрос 8.1.2.** Почему код коллеги не изменил `df`? Как это связано с вопросом `b = a` из первой лекции? Изменилось ли число пропусков в `revenue` после правильного обнуления и при каких условиях оно могло бы измениться?

**Ваш ответ:**

_..._

### 8.2. Условная логика без циклов

**Задание 8.2.1.** Добавьте столбец `size_class` с классом заказа по выручке: `"крупный"` - от 20 000 руб., `"средний"` - от 5 000 руб., `"мелкий"` - все остальное, а для заказов с неизвестной выручкой - `"нет данных"`. Используйте `np.select`, результат приведите к типу `category`. Выведите количество заказов в каждом классе.

**Задание 8.2.2.** Добавьте столбец `region` со значением `"Москва"` для московских заказов и `"регионы"` для остальных (`np.where`), тоже с типом `category`.

> **Подсказка.** Подумайте, в каком порядке перечислять условия, и что произойдет со строкой, где выручка - `NaN`, если отдельного условия для нее не будет.

In [ ]:
# Ваш код

In [ ]:
# Ваш код

**Вопрос 8.2.** Что получилось бы, если перечислить условия в `np.select` в порядке «≥ 5 000», потом «≥ 20 000»? А если убрать условие для пропусков?

**Ваш ответ:**

_..._

---
## Часть 9. Контроль результата

### 9.1. Типы

**Задание 9.1.** Создайте словарь `expected_types`: *столбец -> ожидаемый тип* для всех столбцов, которые вы уже обработали (включая новые). Циклом проверьте каждый столбец `df` и выведите строку вида `price: float64 ✔` или `price: object ✘ (ожидался float64)`. Для столбцов, которых нет в словаре, выведите пометку «еще не обработан».

> **Подсказка.** Тип столбца можно получить как строку через `str(df[col].dtype)`. Точность дат в разных версиях pandas может отличаться (`datetime64[ns]` или `datetime64[us]`), поэтому удобно проверять, что строка **начинается** с ожидаемого значения (метод строк `.startswith`).

In [ ]:
# Ваш код

### 9.2. Пропуски до и после

**Задание 9.2.** Соберите DataFrame `na_check` из двух Series: количество пропусков по столбцам в `raw` («до») и в `df` («после»). Добавьте столбец с разницей. Выведите таблицу.

In [ ]:
# Ваш код

**Вопрос 9.2.** (а) Почему в таблице есть строки с `NaN` в одном из столбцов? (б) Объясните **каждое** изменение числа пропусков: откуда оно взялось и оправдано ли. (в) Почему такая таблица - обязательный шаг после преобразований с `coerce` и `.map`?

**Ваш ответ:**

_..._

---
## Часть 10. Итоговое самостоятельное задание

До сих пор вы шли по шагам: столбец за столбцом, с подсказками, в «живой» таблице `df`. В реальной работе очистка должна быть **воспроизводимой**: когда придет выгрузка за май, ее нужно обработать одной командой, получив тот же результат.

Кроме того, три столбца - `delivery_date`, `weight_kg` и `rating` - вы еще не обрабатывали. С ними вам предстоит разобраться **самостоятельно**: найти проблемы, объяснить их причины и выбрать способ преобразования.

### 10.1. Исследование необработанных столбцов

**Задание 10.1.** Для каждого из столбцов `delivery_date`, `weight_kg`, `rating` исходной таблицы `raw` выясните: какие значения в нем встречаются, какие мешают преобразованию и что они означают по смыслу (опирайтесь на описание данных). Для дат проверьте формат и отдельно посмотрите на значения, которые не являются датами.

In [ ]:
# delivery_date

In [ ]:
# weight_kg

In [ ]:
# rating

**Вопрос 10.1.** Опишите найденные проблемы в трех столбцах и решения, которые вы приняли.

**Ваш ответ:**

_..._

### 10.2. Функция очистки

**Задание 10.2.** Напишите функцию `load_orders(path)`, которая:

1. читает **исходный** CSV (не используйте `df` из предыдущих частей - только файл!);
2. делает `order_id` индексом;
3. приводит **все 13** остальных столбцов к типам, соответствующим их смыслу (включая `delivery_date`, `weight_kg`, `rating`), не теряя ни одного реального значения;
4. добавляет столбцы `revenue` (с нулевой выручкой у отмененных и возвращенных заказов, тип `float64`) и `size_class`;
5. возвращает **кортеж** из двух объектов: очищенного DataFrame и словаря-отчета `report`, где для каждого исходного столбца указано, сколько **непустых** в файле значений стали пропусками после преобразования.

Использовать словари `city_map`, `status_map` и функции из предыдущих частей можно. Номера строк и конкретные значения «вручную» не используйте - функция должна работать и на следующей выгрузке с теми же проблемами.

Затем вызовите функцию: `orders, report = load_orders(DATA_PATH)`, выведите `orders.info()` и отчет в виде строк `price: потеряно 2`.

In [ ]:
def load_orders(path):
    ...
    # return clean_df, report


# orders, report = load_orders(DATA_PATH)

### 10.3. Проверка результата

**Задание 10.3.**

1. **Типы.** Дополните словарь ожидаемых типов тремя новыми столбцами (без `region`, которого нет в `orders`) и проверьте им все столбцы `orders`, как в 9.1. Все проверки должны пройти.
2. **Отчет о потерях.** Для каждого столбца с ненулевыми потерями объясните в ответе, какие исходные значения стали пропусками и почему это оправдано.
3. **Сверка с пошаговым решением.** Убедитесь, что для столбцов `order_date`, `price`, `quantity`, `discount`, `revenue` результат функции совпадает с `df` из частей 5-8 (совпадением считайте и совпадение пропусков).
4. **Логические проверки.** Найдите и выведите номера заказов, у которых:
   - дата доставки раньше даты заказа;
   - статус «доставлен», но дата доставки отсутствует;
   - статус «в пути» или «отменен», но дата доставки есть;
   - оценка вне диапазона 1-5, количество или цена не положительны.

In [ ]:
# Ваш код

In [ ]:
# Ваш код

In [ ]:
# Ваш код

In [ ]:
# Ваш код

**Вопрос 10.3.** Какие дефекты данных обнаружили логические проверки? Можно ли их исправить средствами анализа или нужно сообщить владельцам CRM? Как вы поступите с этими заказами при расчете показателей?

**Ваш ответ:**

_..._

---
### Перед сдачей

- Проверьте, что ноутбук выполняется сверху вниз без ошибок.
- На все вопросы даны ответы своими словами.
- Исходный CSV не изменен.
- Вы можете объяснить любую строку своего кода.